# Representative input for ONNX / INT8 calibration

**Environment:** activate the project conda env with:

```bash
source /gpfs/mnt/gpfs01/usfcc/pusharma/Haider_LDRD/RealTimeAlignment/setEnv.sh
```

This sets `REALTIME_ALIGNMENT_ROOT` and the `qat` environment (see that script for conda and `DATAROOT`).

The MLP exported from `onnx_quantized/qat_onnx.ipynb` expects a single tensor named **`input`** with shape **`(batch_size, 50, 6)`**:

- **50** — `num_particles` (see `onnx_quantized/config.yaml`)
- **6** — `in_features` (three detector readout coordinates × 2, flattened)

This folder includes a small **synthetic** dataset (`data/calibration_samples.npz`) you can use as representative data when running post-training quantization tools, or to sanity-check `model_fp32.onnx` with ONNX Runtime.

In [ ]:
from pathlib import Path
import os
import numpy as np

# If set by: source .../RealTimeAlignment/setEnv.sh
_env_root = os.environ.get("REALTIME_ALIGNMENT_ROOT")
if _env_root:
    ROOT = Path(_env_root).resolve()
    HERE = ROOT / "forAkshay"
    DATA_PATH = HERE / "data" / "calibration_samples.npz"
    ONNX_FP32 = ROOT / "onnx_quantized" / "model_fp32.onnx"
else:
    _cwd = Path.cwd()
    if (_cwd / "data" / "calibration_samples.npz").is_file():
        HERE = _cwd
    elif (_cwd / "forAkshay" / "data" / "calibration_samples.npz").is_file():
        HERE = _cwd / "forAkshay"
    else:
        HERE = _cwd
    ROOT = HERE.parent if HERE.name == "forAkshay" else HERE
    DATA_PATH = HERE / "data" / "calibration_samples.npz"
    ONNX_FP32 = ROOT / "onnx_quantized" / "model_fp32.onnx"

INPUT_NAME = "input"
EXPECTED_FEATURE_DIM = 6
EXPECTED_NUM_PARTICLES = 50

print("HERE =", HERE.resolve())
print("ONNX_FP32 =", ONNX_FP32.resolve())

In [ ]:
with np.load(DATA_PATH) as z:
    inputs = np.array(z["inputs"], dtype=np.float32)
    n_part = int(z["num_particles"]) if "num_particles" in z.files else EXPECTED_NUM_PARTICLES
    n_feat = int(z["in_features"]) if "in_features" in z.files else EXPECTED_FEATURE_DIM

assert inputs.ndim == 3, inputs.shape
assert inputs.shape[1] == n_part == EXPECTED_NUM_PARTICLES, (inputs.shape, n_part)
assert inputs.shape[2] == n_feat == EXPECTED_FEATURE_DIM, (inputs.shape, n_feat)

print("Loaded", DATA_PATH)
print("inputs dtype/shape:", inputs.dtype, inputs.shape)
print("min/max:", float(inputs.min()), float(inputs.max()))

## ONNX Runtime forward pass (optional)

Install if needed: `pip install onnxruntime` (or `onnxruntime-gpu`).

In [ ]:
def run_onnx_batch(onnx_path: Path, x: np.ndarray) -> np.ndarray:
    import onnxruntime as ort
    if not onnx_path.is_file():
        raise FileNotFoundError(f"Missing ONNX model: {onnx_path}")
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    in_name = sess.get_inputs()[0].name
    out = sess.run(None, {in_name: x.astype(np.float32)})
    return out[0]


if ONNX_FP32.is_file():
    y = run_onnx_batch(ONNX_FP32, inputs[:2])
    print("Output shape (batch=2):", y.shape)
else:
    print("Skip inference — place model_fp32.onnx at:", ONNX_FP32)

In [ ]:
# Iterator for calibration / profiling (yields dicts suitable for ORT QDQ or external tools)
def representative_samples():
    for i in range(inputs.shape[0]):
        x = inputs[i : i + 1]  # (1, 50, 6)
        yield {INPUT_NAME: x}


n = sum(1 for _ in representative_samples())
print("Number of single-batch samples:", n)

## Regenerate synthetic data (optional)

Real deployment should use tensors drawn from the same distribution as training readouts. Uncomment to overwrite `data/calibration_samples.npz`.

In [ ]:
# rng = np.random.default_rng(42)
# n = 16
# inputs_new = rng.standard_normal((n, EXPECTED_NUM_PARTICLES, EXPECTED_FEATURE_DIM), dtype=np.float32) * 0.35
# np.savez_compressed(DATA_PATH, inputs=inputs_new,
#                     num_particles=np.int32(EXPECTED_NUM_PARTICLES),
#                     in_features=np.int32(EXPECTED_FEATURE_DIM))
# print("Saved", DATA_PATH)